# LightCyto — fresh TinyViT + custom attention + KD + PPO

**One readable notebook: image paths → grouped split → teacher → distilled student → fusion → PPO → validation lock → final test → deployment.**

This starts a new experiment. No old v2/v4/v5/v6 checkpoints, caches, split CSV, compressed code or embedded dataset is required. It trains from public pretrained initialization, not random initialization: ConvNeXt-Tiny uses ImageNet-1K; TinyViT-5M uses ImageNet-22K distillation followed by ImageNet-1K fine-tuning. The SIPaKMeD teacher is then trained here and frozen for student KD.

**Your original three-query/four-head morphology attention is retained.** The new extension shares the same frozen student encoder for local crops; a residual head learns all observed crop subsets; PPO learns when to stop. This is a fresh implementation and split, not a numerical reproduction of the earlier 97.61% validation result. Older checkpoints are deliberately not loaded.

**Default:** one predeclared seed (42), all stages enabled, final test enabled. For three-seed replication set `seeds=(42,123,2026)` before starting. No 97–99% result is promised. TinyViT parameter count, exported size, accuracy, macro-F1 and real selective inference latency are measured by the run.

## 1. Kaggle setup — only the image dataset is needed

1. Create a notebook, select a **GPU accelerator**, and enable **Internet** for the pinned timm package and official pretrained weights.
2. Attach your original SIPaKMeD dataset with the five `im_*` folders and their `CROPPED` images.
3. In the configuration cell, leave `dataset_root=''` for discovery, or paste the exact folder containing all five class folders. Both flat and nested Kaggle mounts are supported by discovery.
4. Choose seeds and training budgets **before the first run**, then **Run All**.

Typical root from your earlier project: `/kaggle/input/datasets/prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed`. Use the path actually mounted in your notebook; the dataset display title is not necessarily its filesystem path.

`run_final_test=True` already enables final testing. Testing waits for teacher, student, fusion and PPO completion and a validation-only design lock. If a session pauses, the notebook does not report an unfinished study as a final result.

**GPU/session limits:** the code uses one GPU. Epoch logs include a measured estimate for the remaining stage. Save the **full Output folder**, not just the reports ZIP, before a Kaggle session ends.

In [ ]:
import importlib.metadata, importlib.util, subprocess, sys
PINNED_TIMM = '1.0.22'
try:
    installed = importlib.metadata.version('timm')
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != PINNED_TIMM:
    if 'timm' in sys.modules:
        raise RuntimeError('Restart the kernel before changing the imported timm version.')
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-deps','timm=='+PINNED_TIMM])
missing=[name for name in ('huggingface_hub','safetensors') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*missing])
# Kaggle supplies torch/torchvision. This notebook does not replace their CUDA builds.


## 2. Imports
The code is visible and editable. No executable code or metadata is decoded from an opaque payload.

In [ ]:
from __future__ import annotations
import os, sys, re, json, math, time, copy, random, hashlib, shutil, zipfile, gc, inspect
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Categorical
import torchvision
import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, confusion_matrix


## 3. Choose the experiment once

Edit `CFG = Config(...)` below. For example:

```python
CFG = Config(
    dataset_root='',
    seeds=(42,),
    output_root='/kaggle/working/lightcyto_tinyvit_end_to_end',
    resume_source='',
    run_final_test=True,
)
```

| Stage | Default budget | Trained parameters |
|---|---:|---|
| Teacher | 12 epochs | ConvNeXt-Tiny + five-class head |
| Student | 45 epochs per seed | TinyViT + custom attention + classifier |
| Fusion | 30 epochs per seed | Small residual fusion head only |
| PPO | 30 rounds per seed | Small actor and critic only |

Student objective: `CE(label_smoothing=0.05) + 0.25 × T² KL(teacher_T || student_T)`, with `T=2`. RKD, EMA, TTA and ensembles are not part of this fixed recipe. Both teacher and student see the same augmented training view for KD.

`pretrained=True` is the intended scientific setting. `pretrained=False` exists for software smoke tests, not as an equivalent training recipe. Keep the final-test setting enabled if you want a single Run All workflow. Changing scientific settings or code after starting requires a new output directory.

In [ ]:
@dataclass
class Config:
    dataset_root: str = ''  # Blank: find the SIPaKMeD class folders under /kaggle/input.
    output_root: str = '/kaggle/working/lightcyto_tinyvit_end_to_end'
    resume_source: str = ''  # Optional: this notebook's FULL saved output directory.
    seeds: tuple = (42,)  # For replication, choose (42,123,2026) BEFORE the first run.
    split_seed: int = 42
    teacher_seed: int = 7001
    image_size: int = 224
    feature_dim: int = 192
    attention_heads: int = 4
    batch_size: int = 16
    teacher_batch_size: int = 8
    num_workers: int = 0
    teacher_epochs: int = 12
    student_epochs: int = 45
    fusion_epochs: int = 30
    ppo_rounds: int = 30
    teacher_lr: float = 1e-4
    student_lr: float = 3e-4
    fusion_lr: float = 3e-4
    ppo_lr: float = 2e-4
    weight_decay: float = 1e-4
    label_smoothing: float = .05
    kd_weight: float = .25
    kd_temperature: float = 2.
    crop_fraction: float = .60
    overlap_penalty: float = .50
    crop_cost: float = .025
    max_crops: int = 3
    ppo_epochs: int = 4
    ppo_batch_size: int = 256
    ppo_minibatch: int = 128
    ppo_clip: float = .20
    gae_lambda: float = .95
    gamma: float = 1.0  # Finite crop episode; CE-difference rewards telescope.
    entropy_weight: float = .01
    value_weight: float = .5
    target_kl: float = .02
    confidence_threshold: float = .80
    grad_clip: float = .5
    pretrained: bool = True
    amp: bool = True
    allow_cpu: bool = False
    session_hours: float = 10.
    run_final_test: bool = True  # Automatically runs AFTER validation-only design lock.
    benchmark_images: int = 32

CFG = Config()
CLASSES = ('Superficial-Intermediate','Parabasal','Metaplastic','Koilocytotic','Dyskeratotic')
STUDENT_NAME = 'tiny_vit_5m_224.dist_in22k_ft_in1k'
METHODS = ('GLOBAL','DARKNESS','COLOUR','BOUNDARY','ALL_CROPS','CONFIDENCE','PPO')
VERSION = 'tinyvit-kd-shared-crops-ppo-fresh-v1'
MEAN = torch.tensor([.485,.456,.406]).view(1,3,1,1)
STD = torch.tensor([.229,.224,.225]).view(1,3,1,1)


## 4. Checkpoints, session budget and selection

Every stage saves `best.pt`, `last.pt`, `history.csv` and `complete.json`. `last.pt` includes AdamW and AMP-scaler state. Resume replays an interrupted partial epoch; completed epochs are reused. Random order and augmentation are seeded by the absolute epoch. Exact bitwise reproducibility on a different GPU/runtime is not promised.

Best checkpoints use validation macro-F1, then accuracy, then fewer crops when applicable, then lower CE. Fusion uses the mean across its seven nonempty subsets. The test set never selects an epoch or method.

A session pause is checked at epoch/round and cache-batch boundaries. The current epoch can exceed the chosen wall-time budget; leave a margin before Kaggle's hard session limit.

In [ ]:
class SessionPause(RuntimeError): pass

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def json_value(x):
    if isinstance(x, dict): return {str(k):json_value(v) for k,v in x.items()}
    if isinstance(x, (tuple,list)): return [json_value(v) for v in x]
    if isinstance(x, (np.integer,np.floating)): return x.item()
    if isinstance(x, Path): return str(x)
    if isinstance(x, torch.Tensor): return x.detach().cpu().tolist()
    return x

def save_json(x, path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(json_value(x),indent=2,allow_nan=False)); os.replace(tmp,path)

def read_json(path): return json.loads(Path(path).read_text())
def object_hash(x): return hashlib.sha256(json.dumps(json_value(x),sort_keys=True).encode()).hexdigest()
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def save_pt(x,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');torch.save(x,tmp);os.replace(tmp,path)

def load_pt(path): return torch.load(path,map_location='cpu',weights_only=True)
def cpu_state(model): return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
def clear_bias_cache(model):
    for m in model.modules():
        if hasattr(m,'attention_bias_cache'): m.attention_bias_cache.clear()

def install_state(model,state):
    model.load_state_dict(state,strict=True);clear_bias_cache(model)

def budget(ctx):
    if time.monotonic() >= ctx['deadline']: raise SessionPause('Session budget reached; resume this full output.')

def metric_values(y,logits,crops=None):
    y=np.asarray(y,dtype=int);logits=np.asarray(logits,dtype=np.float64)
    assert len(y)>0 and logits.shape==(len(y),5) and np.isfinite(logits).all()
    p=np.exp(logits-logits.max(1,keepdims=True));p/=p.sum(1,keepdims=True);pred=p.argmax(1)
    result={'n':len(y),'correct':int((y==pred).sum()),'accuracy':float(accuracy_score(y,pred)),
            'macro_f1':float(f1_score(y,pred,labels=list(range(5)),average='macro',zero_division=0)),
            'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'ce':float(-np.log(p[np.arange(len(y)),y].clip(1e-12)).mean())}
    if crops is not None: result['mean_crops']=float(np.mean(crops))
    return result

def quality(m): return (m['macro_f1'],m['accuracy'],-m.get('mean_crops',0.),-m['ce'])
def freeze(model):
    model.eval()
    for p in model.parameters():p.requires_grad_(False)
    return model

def fit_stage(ctx,name,model,epochs,lr,seed,train_epoch,validate,amp=False):
    """Atomic epoch-boundary resume. An interrupted partial epoch is replayed."""
    out=ctx['root']/name;out.mkdir(parents=True,exist_ok=True)
    last,best,complete=out/'last.pt',out/'best.pt',out/'complete.json'
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=CFG.weight_decay)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    start=0;history=[];best_key=(-1.,);best_epoch=0
    if last.exists():
        ck=load_pt(last)
        if ck['contract']!=ctx['contract_hash']:raise ValueError('Checkpoint contract mismatch: '+name)
        install_state(model,ck['model']);opt.load_state_dict(ck['optimizer']);scaler.load_state_dict(ck['scaler'])
        start=int(ck['epoch']);history=ck['history'];best_key=tuple(ck['best_key']);best_epoch=int(ck['best_epoch'])
    if complete.exists():
        receipt=read_json(complete)
        if not best.exists() or start!=epochs or receipt['best_sha256']!=file_hash(best):
            raise ValueError('Incomplete or modified saved stage: '+name)
        install_state(model,load_pt(best)['model']);return model
    for epoch in range(start+1,epochs+1):
        budget(ctx);seed_all(seed+epoch*1009)
        epoch_lr=lr*(.1+.9*.5*(1+math.cos(math.pi*(epoch-1)/max(1,epochs))))
        for g in opt.param_groups:g['lr']=epoch_lr
        t=time.monotonic();train_info=train_epoch(model,opt,scaler,epoch)
        model.eval();clear_bias_cache(model);m=validate(model)
        row={'epoch':epoch,'lr':epoch_lr,**train_info,**m,'seconds':time.monotonic()-t}
        history.append(row);key=quality(m)
        if key>best_key:
            best_key=key;best_epoch=epoch
            save_pt({'model':cpu_state(model),'epoch':epoch,'metrics':m,'contract':ctx['contract_hash']},best)
        save_pt({'model':cpu_state(model),'optimizer':opt.state_dict(),'scaler':scaler.state_dict(),
                 'epoch':epoch,'history':history,'best_key':list(best_key),'best_epoch':best_epoch,
                 'contract':ctx['contract_hash']},last)
        pd.DataFrame(history).to_csv(out/'history.csv',index=False)
        save_json({'status':'training','stage':name,'epoch':epoch,'epochs':epochs},ctx['root']/'session_status.json')
        print(f'{name} {epoch}/{epochs}: val acc={m["accuracy"]:.4%}, F1={m["macro_f1"]:.4%}, {row["seconds"]:.1f}s; stage ETA ~{(epochs-epoch)*row["seconds"]/60:.1f} min',flush=True)
    save_json({'epochs':epochs,'best_epoch':best_epoch,'best_sha256':file_hash(best),
               'contract':ctx['contract_hash']},complete)
    install_state(model,load_pt(best)['model']);return model


## 5. Source-group split and data audit

Only original `CROPPED` cell images are classification examples; full microscope fields are not mixed into the sample list. Names such as `001_03.bmp` are linked to source image `001`. Group IDs include the class folder. If available, exact duplicate parent-field images also join groups. Exact duplicate cell pixels are removed, with their source groups joined before splitting; conflicting duplicate labels stop the run.

The split is deterministic: take one of five stratified **group** folds as test, then one of five remaining-development folds as validation. This targets approximately **64% train / 16% validation / 20% test**; exact counts vary because groups remain intact. The notebook saves the manifest and verifies class coverage, group separation and exact-pixel separation.

This is a new source-image-group holdout. It is **not patient-level validation**; no patient IDs are available. Near-duplicate/rotated-image detection is not implemented. All images may be opened for integrity hashes during the audit; test images are not used for fitting or selection.

Earlier experiments on SIPaKMeD informed this design, so report it as a **post-selection follow-up**, not as an untouched external confirmation. Randomly regenerating a split does not erase earlier dataset exposure.

The same training partition is used for teacher/student, fusion and policy fitting. It is not an independent policy/calibration reserve; validation and test remain separate.

In [ ]:
def resolve_dataset(hint):
    if hint:
        p=Path(hint).expanduser()
        if not p.is_dir():raise FileNotFoundError(f'DATASET_ROOT does not exist: {p}')
        required={'im_'+c for c in CLASSES}
        if not required <= {x.name for x in p.iterdir() if x.is_dir()}:
            raise ValueError('dataset_root must directly contain all five im_* class folders.')
        return p
    root=Path('/kaggle/input');required={'im_'+c for c in CLASSES}
    matches=[]
    for d in root.rglob('im_Dyskeratotic'):
        if d.is_dir() and required <= {p.name for p in d.parent.iterdir() if p.is_dir()}:
            matches.append(d.parent)
    matches=sorted(set(matches))
    if len(matches)!=1:
        raise FileNotFoundError('Set CFG.dataset_root to the folder containing the five im_* class folders. '
                                f'Found {len(matches)} candidates: {matches}')
    return matches[0]

def restore_output():
    root=Path(CFG.output_root).expanduser()
    if CFG.resume_source:
        src=Path(CFG.resume_source).expanduser()
        if not src.is_dir() or not (src/'experiment.json').is_file():
            raise FileNotFoundError('resume_source must be this notebook\'s FULL output with experiment.json, checkpoints and caches.')
        if src.resolve()!=root.resolve():
            if root.exists() and any(root.iterdir()):
                print('Existing working output takes priority over resume_source.',flush=True)
            else:shutil.copytree(src,root,dirs_exist_ok=True)
    root.mkdir(parents=True,exist_ok=True);return root

def scan_cells(root):
    rows=[]
    for p in sorted(root.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in {'.bmp','.png','.jpg','.jpeg','.tif','.tiff'}:continue
        rel=p.relative_to(root)
        if not any(part.upper()=='CROPPED' for part in rel.parts[:-1]):continue
        names={part[3:] for part in rel.parts[:-1] if part.startswith('im_') and part[3:] in CLASSES}
        if len(names)!=1:raise ValueError(f'Cannot identify one class for {rel}')
        cls=next(iter(names));m=re.fullmatch(r'(\d+)[_-](\d+)',p.stem)
        if not m:raise ValueError(f'Unknown source-image filename: {rel}. Supply original SIPaKMeD CROPPED folders; do not randomly split ambiguous crops.')
        with Image.open(p) as im:
            im=im.convert('RGB');arr=np.asarray(im)
            ph=hashlib.sha256(str(im.size).encode()+arr.tobytes()).hexdigest()
        rows.append({'relative_path':rel.as_posix(),'class_name':cls,'label':CLASSES.index(cls),
                     'source_group':f'{cls}:{int(m.group(1))}', 'pixel_sha256':ph,'file_sha256':file_hash(p),
                     'sample_id':hashlib.sha256(rel.as_posix().encode()).hexdigest()[:20]})
    if not rows:raise ValueError('No cell images found under CROPPED directories.')
    frame=pd.DataFrame(rows)
    if set(frame.class_name)!=set(CLASSES):raise ValueError('All five SIPaKMeD classes are required.')
    # Verify available parent-field images as well as cell crops. Repeated source fields
    # under different IDs must stay in the same partition even if their crops differ.
    source_hashes={}
    for group,sub in frame.groupby('source_group',sort=False):
        cell=root/sub.iloc[0].relative_path
        stem=re.fullmatch(r'(\d+)[_-](\d+)',cell.stem).group(1)
        candidates=[x for x in cell.parent.parent.iterdir() if x.is_file() and x.stem==stem
                    and x.suffix.lower() in {'.bmp','.png','.jpg','.jpeg','.tif','.tiff'}]
        hashes=[]
        for field in candidates:
            with Image.open(field) as im:
                im=im.convert('RGB');hashes.append(hashlib.sha256(str(im.size).encode()+np.asarray(im).tobytes()).hexdigest())
        if len(set(hashes))>1:raise ValueError('Ambiguous source-field files for '+group)
        source_hashes[group]=hashes[0] if hashes else ''
    frame['source_image_sha256']=frame.source_group.map(source_hashes)
    # Union source-image groups connected by exact duplicate pixels, then deduplicate.
    parent={x:x for x in frame.source_group.unique()}
    def find(x):
        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]
        return x
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b:parent[max(a,b)]=min(a,b)
    for _,sub in frame[frame.source_image_sha256!=''].groupby('source_image_sha256',sort=False):
        groups=list(sub.source_group.unique())
        for g in groups[1:]:union(groups[0],g)
    for _,sub in frame.groupby('pixel_sha256',sort=False):
        if sub.label.nunique()>1:raise ValueError('Identical pixels have conflicting labels; resolve the dataset before training.')
        groups=list(sub.source_group.unique())
        for g in groups[1:]:union(groups[0],g)
    frame['group_id']=frame.source_group.map(lambda x:find(x))
    duplicate_rows=frame[frame.duplicated('pixel_sha256',keep='first')].copy()
    frame=frame.drop_duplicates('pixel_sha256',keep='first').reset_index(drop=True)
    return frame,duplicate_rows

def assign_partitions(frame):
    frame=frame.copy();frame['split']='train'
    outer=StratifiedGroupKFold(5,shuffle=True,random_state=CFG.split_seed)
    dev,test=next(outer.split(frame,frame.label,frame.group_id))
    inner=StratifiedGroupKFold(5,shuffle=True,random_state=CFG.split_seed+1)
    inner_frame=frame.iloc[dev]
    _,val=next(inner.split(inner_frame,inner_frame.label,inner_frame.group_id))
    frame.loc[test,'split']='test';frame.loc[dev[val],'split']='val'
    assert frame.groupby('group_id').split.nunique().max()==1
    assert frame.groupby('pixel_sha256').split.nunique().max()==1
    for part in ('train','val','test'):
        if set(frame[frame.split==part].label)!=set(range(5)):raise ValueError('A partition lacks a class; review source groups.')
    return frame

def implementation_hash():
    # Functions/methods are source-resolvable in IPython; whole interactive classes may not be.
    definitions={}
    for name,obj in globals().copy().items():
        if name=='Config':continue  # Scientific config values are hashed separately.
        if getattr(obj,'__module__',None)!=__name__:continue
        try:
            if inspect.isfunction(obj):definitions[name]=inspect.getsource(obj).strip()
            elif inspect.isclass(obj):
                methods={}
                for key,value in vars(obj).items():
                    if isinstance(value,(staticmethod,classmethod)):value=value.__func__
                    if inspect.isfunction(value):methods[key]=inspect.getsource(value).strip()
                definitions[name]={'bases':[base.__name__ for base in obj.__bases__],'methods':methods}
        except (OSError,TypeError) as e:
            raise RuntimeError('Run the visible notebook definition cells normally before prepare().') from e
    return object_hash(definitions)

def prepare():
    if CFG.image_size!=224 or CFG.max_crops!=3:raise ValueError('This frozen recipe uses 224px and exactly three available crop actions.')
    if not CFG.seeds or len(set(CFG.seeds))!=len(CFG.seeds):raise ValueError('Seeds must be a nonempty unique tuple.')
    if min(CFG.teacher_epochs,CFG.student_epochs,CFG.fusion_epochs,CFG.ppo_rounds)<1:raise ValueError('All stages need positive budgets.')
    root=restore_output();dataset=resolve_dataset(CFG.dataset_root)
    scanned,duplicates=scan_cells(dataset);frame=assign_partitions(scanned)
    fingerprint=object_hash(frame.to_dict('records'))
    scientific=asdict(CFG)
    for k in ('dataset_root','output_root','resume_source','session_hours','run_final_test','allow_cpu','benchmark_images'):scientific.pop(k)
    runtime={'torch':str(torch.__version__),'torchvision':str(torchvision.__version__),'timm':timm.__version__,
             'numpy':np.__version__,'pandas':pd.__version__}
    contract={'version':VERSION,'implementation_sha256':implementation_hash(),'config':scientific,'dataset_sha256':fingerprint,'runtime':runtime,
              'classes':CLASSES,'teacher':'torchvision:convnext_tiny ImageNet1K_V1',
              'student':STUDENT_NAME,'grouping':'source-image groups + exact duplicate union; patient IDs unverified',
              'evaluation_scope':'post-selection follow-up on a previously studied dataset; new source-group holdout'}
    ch=object_hash(contract);contract['contract_hash']=ch
    if (root/'experiment.json').exists():
        if read_json(root/'experiment.json')['contract_hash']!=ch:
            raise ValueError('Dataset, settings, runtime or version changed. Restore the original settings or choose a NEW output_root.')
    else:save_json(contract,root/'experiment.json')
    frame.to_csv(root/'manifest.csv',index=False);duplicates.to_csv(root/'duplicates_removed.csv',index=False)
    summary=frame.groupby(['split','class_name']).agg(images=('sample_id','size'),groups=('group_id','nunique')).reset_index()
    summary.to_csv(root/'split_summary.csv',index=False)
    save_json({'raw_cells':len(scanned)+len(duplicates),'unique_cells':len(frame),'duplicates_removed':len(duplicates),
               'source_groups_disjoint':True,'patient_grouping_verified':False,
               'cells_with_verified_source_field':int((frame.source_image_sha256!='').sum()),
               'near_duplicate_detection':False,
               'test_pixel_access':'Hashing for integrity only during audit; model evaluation follows design lock.'},root/'data_audit.json')
    frame['path']=frame.relative_path.map(lambda x:str(dataset/x))
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type=='cpu' and not CFG.allow_cpu:raise RuntimeError('Enable a Kaggle GPU. CPU training is intentionally disabled by default.')
    ctx={'root':root,'dataset':dataset,'frame':frame,'contract_hash':ch,'device':device,
         'deadline':time.monotonic()+CFG.session_hours*3600}
    for part in ('train','val','test'):ctx[part]=frame[frame.split==part].reset_index(drop=True)
    print('Dataset:',dataset,'\nDevice:',device,'\nPartitions:',frame.split.value_counts().to_dict(),flush=True)
    return ctx


## 6. Teacher and distilled global student

**Student:** RGB → aspect-preserving white letterbox + valid mask → TinyViT-5M feature map → 192-channel projection → three learned queries / four attention heads → 192-dimensional feature → five-class head.

The attention block retains the earlier learned spatial scorer, log-prior bias, positional keys, cross-attention and residual global pooling. It uses darkness, colour/intensity and local intensity-change cues. Those maps are **weak image cues, not verified nucleus/cytoplasm/boundary masks**.

**Teacher:** ImageNet-initialized ConvNeXt-Tiny is trained on the training partition. Its validation-selected checkpoint is frozen for KD; it is not included at deployment. Public TinyViT pretraining distillation and our SIPaKMeD teacher–student KD are two different training stages and should both be disclosed.

Training augmentation uses flips, right-angle rotations and mild brightness/colour changes. Validation/test preprocessing is deterministic. No stain-normalization transform is fitted to test data.

In [ ]:
def augment_pil(im):
    if random.random()<.5: im=ImageOps.mirror(im)
    if random.random()<.5: im=ImageOps.flip(im)
    im=im.rotate(90*random.randrange(4),expand=True)
    im=ImageEnhance.Color(im).enhance(random.uniform(.92,1.08))
    im=ImageEnhance.Brightness(im).enhance(random.uniform(.94,1.06))
    return im

def letterbox(im,size):
    w,h=im.size; scale=min(size/w,size/h)
    nw,nh=max(1,round(w*scale)),max(1,round(h*scale)); left=(size-nw)//2; top=(size-nh)//2
    canvas=Image.new('RGB',(size,size),(255,255,255))
    canvas.paste(im.resize((nw,nh),Image.Resampling.BILINEAR),(left,top))
    rgb=torch.from_numpy(np.asarray(canvas).copy()).permute(2,0,1).float()/255.
    valid=torch.zeros(1,size,size); valid[:,top:top+nh,left:left+nw]=1.
    meta=(left,top,nw,nh,w,h)
    return rgb,valid,meta

def morphology_proxies(rgb,valid):
    # Image-derived proxies, not ground-truth segmentation or verified anatomy.
    gray=.299*rgb[:,0:1]+.587*rgb[:,1:2]+.114*rgb[:,2:3]
    sat=rgb.amax(1,keepdim=True)-rgb.amin(1,keepdim=True)
    nucleus=(1-gray).clamp_min(.01).pow(2)
    cyto=(sat+.03)*(gray+.10)
    dx=F.pad((gray[:,:,:,1:]-gray[:,:,:,:-1]).abs(),(0,1,0,0))
    dy=F.pad((gray[:,:,1:,:]-gray[:,:,:-1,:]).abs(),(0,0,0,1))
    interior=(F.avg_pool2d(valid,5,1,2)>.99).to(rgb.dtype)
    edge=(dx+dy+.005)*interior
    prior=torch.cat([nucleus,cyto,edge],1)*valid
    return prior.clamp_min(1e-8)



class ConvGN(nn.Sequential):
    def __init__(self,a,b,k=1,s=1,groups=1,act=True):
        gn=math.gcd(8,b)
        super().__init__(nn.Conv2d(a,b,k,s,k//2,groups=groups,bias=False),nn.GroupNorm(gn,b),
                         nn.SiLU(inplace=False) if act else nn.Identity())


class QueryMorphologyAttention(nn.Module):
    """Three learned morphology queries, with image-derived proxy biases.

    Uses original encoder resolution. The scorer and four attention heads are learned;
    proxy maps are weak cues, not nucleus/cytoplasm ground-truth masks.
    """
    def __init__(self, c, heads):
        super().__init__()
        self.heads, self.c = heads, c
        self.score = nn.Sequential(nn.Conv2d(c, max(4,c//4), 1), nn.GELU(), nn.Conv2d(max(4,c//4),3,1))
        self.prior_strength = nn.Parameter(torch.zeros(3))
        self.queries = nn.Parameter(torch.randn(3,c)*.02)
        self.position = nn.Linear(2,c,bias=False)
        self.token_norm = nn.LayerNorm(c)
        self.query_norm = nn.LayerNorm(c)
        self.cross = nn.MultiheadAttention(c,heads,batch_first=True,dropout=0.)
        self.region_norm = nn.LayerNorm(c)
        self.channel_mix = nn.Linear(3*c,c)
    def forward(self, f, prior, valid):
        b,c,h,w = f.shape
        mask = F.adaptive_avg_pool2d(valid,(h,w)).flatten(1) > .05
        if not mask.any(1).all(): raise ValueError('No valid image tokens')
        yy,xx = torch.meshgrid(torch.linspace(-1,1,h,device=f.device,dtype=f.dtype),
                               torch.linspace(-1,1,w,device=f.device,dtype=f.dtype),indexing='ij')
        xy = torch.stack((xx,yy),-1).reshape(1,h*w,2)
        tokens = f.flatten(2).transpose(1,2)
        keys = self.token_norm(tokens+self.position(xy))
        p = F.adaptive_avg_pool2d(prior,(h,w)).clamp_min(1e-8)
        bias = (self.score(f)+F.softplus(self.prior_strength)[None,:,None,None]*p.log()).flatten(2)
        bias = bias.masked_fill(~mask[:,None,:],-1e4)
        bias = bias[:,None].expand(-1,self.heads,-1,-1).reshape(b*self.heads,3,h*w)
        queries = self.query_norm(self.queries)[None].expand(b,-1,-1)
        regions,weights = self.cross(queries,keys,tokens,attn_mask=bias,need_weights=True)
        pooled = (tokens*mask[:,:,None]).sum(1)/mask.sum(1,keepdim=True)
        z = pooled + .25*self.channel_mix(self.region_norm(regions).flatten(1))
        return z,weights.reshape(b,3,h,w)


class Cells(Dataset):
    def __init__(self,frame,training=False):self.frame=frame.reset_index(drop=True);self.training=training
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        with Image.open(r.path) as im:im=im.convert('RGB')
        if self.training:im=augment_pil(im)
        rgb,valid,meta=letterbox(im,CFG.image_size)
        return {'rgb':rgb,'valid':valid,'raw':np.asarray(im).copy(),'meta':meta,'y':int(r.label),'id':str(r.sample_id)}

def collate(items):
    return {'rgb':torch.stack([x['rgb'] for x in items]),'valid':torch.stack([x['valid'] for x in items]),
            'y':torch.tensor([x['y'] for x in items]),'raw':[x['raw'] for x in items],
            'meta':[x['meta'] for x in items],'ids':[x['id'] for x in items]}

def loader(frame,training=False,batch_size=None):
    return DataLoader(Cells(frame,training),batch_size=batch_size or CFG.batch_size,shuffle=training,
                      num_workers=CFG.num_workers,collate_fn=collate,pin_memory=torch.cuda.is_available())

def normalize_teacher(x):return (x-MEAN.to(x))/STD.to(x)

class Student(nn.Module):
    def __init__(self,pretrained=False):
        super().__init__()
        self.backbone=timm.create_model(STUDENT_NAME,pretrained=pretrained,num_classes=0)
        self.backbone.head=nn.Identity()
        self.register_buffer('mean',torch.tensor(self.backbone.pretrained_cfg['mean']).view(1,3,1,1))
        self.register_buffer('std',torch.tensor(self.backbone.pretrained_cfg['std']).view(1,3,1,1))
        self.projection=ConvGN(self.backbone.num_features,CFG.feature_dim)
        self.attention=QueryMorphologyAttention(CFG.feature_dim,CFG.attention_heads)
        self.norm=nn.LayerNorm(CFG.feature_dim)
        self.head=nn.Sequential(nn.Dropout(.15),nn.Linear(CFG.feature_dim,5))
    def forward(self,rgb,valid):
        f=self.backbone.forward_features((rgb-self.mean.to(rgb))/self.std.to(rgb))
        if f.ndim!=4:raise ValueError('TinyViT must return NCHW features: '+str(f.shape))
        z,maps=self.attention(self.projection(f),morphology_proxies(rgb,valid),valid)
        z=self.norm(z);return z,self.head(z),maps

def new_teacher(pretrained):
    from torchvision.models import convnext_tiny,ConvNeXt_Tiny_Weights
    m=convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
    m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,5)
    return m

def kd_loss(student,teacher):
    t=CFG.kd_temperature
    return F.kl_div(F.log_softmax(student.float()/t,1),F.softmax(teacher.detach().float()/t,1),reduction='batchmean')*t*t

@torch.no_grad()
def image_validation(model,frame,device,is_teacher=False):
    model.eval();ys=[];zs=[]
    for b in loader(frame,batch_size=CFG.teacher_batch_size if is_teacher else CFG.batch_size):
        x,v=b['rgb'].to(device),b['valid'].to(device)
        z=model(normalize_teacher(x)) if is_teacher else model(x,v)[1]
        ys.extend(b['y'].tolist());zs.append(z.float().cpu().numpy())
    return metric_values(ys,np.concatenate(zs))

def image_train_epoch(ctx,is_teacher=False,teacher=None):
    device=ctx['device'];use_amp=CFG.amp and device.type=='cuda'
    def step(model,opt,scaler,epoch):
        model.train();clear_bias_cache(model);total=ce_sum=kd_sum=0.;n=0
        for b in loader(ctx['train'],True,CFG.teacher_batch_size if is_teacher else CFG.batch_size):
            x,v,y=b['rgb'].to(device),b['valid'].to(device),b['y'].to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type,enabled=use_amp):
                if is_teacher:z=model(normalize_teacher(x));kd=z.new_zeros(())
                else:
                    z=model(x,v)[1]
                    with torch.no_grad():
                        tz=torch.cat([teacher(normalize_teacher(c)) for c in x.split(CFG.teacher_batch_size)])
                    kd=kd_loss(z,tz)
                ce=F.cross_entropy(z.float(),y,label_smoothing=CFG.label_smoothing)
                loss=ce+(0. if is_teacher else CFG.kd_weight)*kd
            if not torch.isfinite(loss):raise FloatingPointError('Non-finite training loss.')
            scaler.scale(loss).backward();scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),2.)
            scaler.step(opt);scaler.update()
            k=len(y);n+=k;total+=float(loss.detach())*k;ce_sum+=float(ce.detach())*k;kd_sum+=float(kd.detach())*k
        return {'train_loss':total/n,'train_ce':ce_sum/n,'train_kd':kd_sum/n}
    return step


## 7. Native-image crops and new feature caches

The global attention maps and weak cues propose darkness, colour and boundary crops. Each crop spans 60% of native image width and height; an overlap penalty discourages repeated regions. No true label enters proposal generation.

The student is frozen after KD. Training/validation features for all three proposals are cached once, using the **same encoder weights**. These are newly generated caches, not the old v4 inputs. They make fusion and PPO fitting cheap without retraining the encoder.

Cached counterfactual features can be used by the training environment to reveal a chosen action's result. The policy state explicitly masks every **unobserved** crop. Real selective inference below extracts and encodes only requested crops. Cached-evaluation runtime is never presented as deployment latency.

In [ ]:
@torch.no_grad()
def crop_boxes(maps,rgb,valid,metas):
    """Weak cue peaks with a diversity penalty. No labels or unseen crop features."""
    size=rgb.shape[-1]
    heat=F.interpolate(maps.float(),(size,size),mode='bilinear',align_corners=False)
    heat=heat*morphology_proxies(rgb.float(),valid.float())*valid
    boxes=[]
    for i,(left,top,nw,nh,ow,oh) in enumerate(metas):
        previous=[];sample=[]
        for a in range(3):
            score=heat[i,a].clone()
            for x0,y0,x1,y1 in previous:score[y0:y1,x0:x1]*=(1-CFG.overlap_penalty)
            score=score.masked_fill(valid[i,0]<.5,-1)
            idx=int(score.argmax());py,px=divmod(idx,size)
            cx=np.clip((px-left+.5)/nw,0,1)*ow;cy=np.clip((py-top+.5)/nh,0,1)*oh
            bw=max(1,min(ow,round(ow*CFG.crop_fraction)));bh=max(1,min(oh,round(oh*CFG.crop_fraction)))
            x0=int(np.clip(round(cx-bw/2),0,ow-bw));y0=int(np.clip(round(cy-bh/2),0,oh-bh))
            sample.append((x0,y0,x0+bw,y0+bh))
            previous.append((max(left,int(left+x0/ow*nw)),max(top,int(top+y0/oh*nh)),
                             min(left+nw,int(math.ceil(left+(x0+bw)/ow*nw))),
                             min(top+nh,int(math.ceil(top+(y0+bh)/oh*nh)))))
        boxes.append(sample)
    return np.asarray(boxes,dtype=np.int64)

def crops_from_batch(batch,boxes,indices,actions,device):
    rgb=[];valid=[]
    for i,a in zip(indices,actions):
        if int(a) not in (1,2,3):raise ValueError('STOP cannot extract a crop.')
        crop=Image.fromarray(batch['raw'][int(i)]).crop(tuple(map(int,boxes[int(i),int(a)-1])))
        x,v,_=letterbox(crop,CFG.image_size);rgb.append(x);valid.append(v)
    return torch.stack(rgb).to(device),torch.stack(valid).to(device)

@torch.no_grad()
def build_cache(ctx,student,seed,part):
    if part=='test' and not (ctx['root']/'design_lock.json').is_file():raise RuntimeError('Test access requires completed validation design lock.')
    out=ctx['root']/f'seed_{seed}'/'cache'/part;out.mkdir(parents=True,exist_ok=True)
    frame=ctx[part];device=ctx['device'];freeze(student)
    signature=object_hash({'contract':ctx['contract_hash'],'student':file_hash(ctx['root']/f'seed_{seed}'/'student'/'best.pt'),
                           'ids':frame.sample_id.tolist(),'crop_version':VERSION})
    chunks=[]
    for j,b in enumerate(loader(frame)):
        budget(ctx);p=out/f'batch_{j:05d}.pt'
        if p.exists():
            chunk=load_pt(p)
            if chunk['signature']!=signature or chunk['ids']!=b['ids']:raise ValueError('Feature cache identity mismatch.')
        else:
            x,v=b['rgb'].to(device),b['valid'].to(device)
            g,base,maps=student(x,v);boxes=crop_boxes(maps,x,v,b['meta']);locals=[]
            for a in range(1,4):
                cx,cv=crops_from_batch(b,boxes,list(range(len(x))),[a]*len(x),device)
                locals.append(student(cx,cv)[0].float().cpu())
            chunk={'g':g.float().cpu(),'base':base.float().cpu(),'local':torch.stack(locals,1),
                   'y':b['y'],'ids':b['ids'],'boxes':torch.from_numpy(boxes),'signature':signature}
            save_pt(chunk,p)
        chunks.append(chunk)
    cache={k:torch.cat([c[k] for c in chunks]) for k in ('g','base','local','y','boxes')}
    cache['ids']=[i for c in chunks for i in c['ids']]
    assert cache['ids']==frame.sample_id.tolist()
    return cache


## 8. Global–local residual fusion

Fusion receives the global feature, three ordered crop-feature slots, and their observed/unobserved mask. Unobserved slots are zero. It learns a residual correction to global logits; an empty subset returns the global prediction exactly.

The head is trained on **all seven nonempty subsets**, with equal subset weight. This supports sequential crop acquisition and prevents a three-crop-only head from being incorrectly reused for one-crop inference. The encoder and original classifier remain frozen. The first residual is zero by construction.

In [ ]:
def subset_mask(bits):
    return ((bits.long().unsqueeze(1) >> torch.arange(3,device=bits.device)) & 1).bool()

class Fusion(nn.Module):
    def __init__(self):
        super().__init__();d=CFG.feature_dim
        self.net=nn.Sequential(nn.Linear(4*d+3,128),nn.LayerNorm(128),nn.GELU(),nn.Linear(128,5))
        nn.init.zeros_(self.net[-1].weight);nn.init.zeros_(self.net[-1].bias)
    def forward(self,g,base,local,mask):
        # Unobserved local evidence is zeroed before the network.
        inp=torch.cat([g,(local*mask[:,:,None]).flatten(1),mask.float()],1)
        delta=self.net(inp)*mask.any(1,keepdim=True)
        return base+delta

def cache_device(cache,device):return {k:(v.to(device) if isinstance(v,torch.Tensor) else v) for k,v in cache.items()}

@torch.no_grad()
def subset_logits(fusion,cache,device):
    fusion.eval();outs=[]
    for start in range(0,len(cache['y']),256):
        c=cache_device({k:v[start:start+256] for k,v in cache.items() if isinstance(v,torch.Tensor)},device)
        per=[]
        for b in range(8):
            mask=subset_mask(torch.full((len(c['y']),),b,device=device))
            per.append(fusion(c['g'],c['base'],c['local'],mask).cpu())
        outs.append(torch.stack(per,1))
    return torch.cat(outs)

def fusion_functions(train,val,device):
    def step(model,opt,scaler,epoch):
        model.train();order=torch.randperm(len(train['y']));total=0.;n=0
        for ix in order.split(128):
            g=train['g'][ix].to(device);base=train['base'][ix].to(device);local=train['local'][ix].to(device);y=train['y'][ix].to(device)
            bits=torch.arange(1,8,device=device).repeat(len(ix));mask=subset_mask(bits)
            z=model(g.repeat_interleave(7,0),base.repeat_interleave(7,0),local.repeat_interleave(7,0),mask)
            loss=F.cross_entropy(z,y.repeat_interleave(7))
            opt.zero_grad(set_to_none=True);loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),2.);opt.step()
            total+=float(loss.detach())*len(ix);n+=len(ix)
        return {'train_loss':total/n}
    def validate(model):
        logits=subset_logits(model,val,device)
        per=[metric_values(val['y'].numpy(),logits[:,b].numpy()) for b in range(1,8)]
        return {k:float(np.mean([m[k] for m in per])) for k in ('accuracy','macro_f1','ce')}
    return step,validate


## 9. PPO-Clip + GAE

| Component | Definition |
|---|---|
| State | Normalized global feature, mean observed local feature, probabilities, entropy, top-two margin, visited mask and crop count |
| State size | 395 values with 192-dimensional features |
| Actor / critic | Separate two-hidden-layer 64-unit tanh MLPs |
| Actions | STOP, DARKNESS, COLOUR, BOUNDARY |
| Invalid actions | Previously visited cue actions are masked |
| Episode | STOP at any point, or terminal after the third distinct cue action |
| Crop reward | `CE_before - CE_after - 0.025` |
| Terminal reward | `+1` correct; `-1` incorrect |
| PPO | Clip 0.20, four update epochs, KL early stop 0.02, gradient norm 0.5 |
| GAE | Lambda 0.95; gamma 1 for the finite crop episode |

Normalization statistics are fitted on training features only. Fresh action trajectories are sampled from the current policy; only frozen image features are reused. Old action log-probabilities and action masks are kept fixed within each PPO update. True labels only calculate training rewards; the actor never receives them. Terminal states are not bootstrapped.

These are predeclared starting settings, not SIPaKMeD-optimal values proved by the PPO papers. The policy may validly learn to STOP frequently when crop evidence is unhelpful.

In [ ]:
class Policy(nn.Module):
    def __init__(self,train_cache=None):
        super().__init__();d=CFG.feature_dim;state_dim=2*d+11
        for name,default in [('g_mean',torch.zeros(d)),('g_std',torch.ones(d)),('l_mean',torch.zeros(d)),('l_std',torch.ones(d))]:
            self.register_buffer(name,default)
        if train_cache is not None:
            self.g_mean.copy_(train_cache['g'].mean(0));self.g_std.copy_(train_cache['g'].std(0,unbiased=False).clamp_min(.001))
            l=train_cache['local'].flatten(0,1)
            self.l_mean.copy_(l.mean(0));self.l_std.copy_(l.std(0,unbiased=False).clamp_min(.001))
        self.actor=nn.Sequential(nn.Linear(state_dim,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh(),nn.Linear(64,4))
        self.critic=nn.Sequential(nn.Linear(state_dim,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh(),nn.Linear(64,1))
        for network in (self.actor,self.critic):
            for layer in network:
                if isinstance(layer,nn.Linear):nn.init.orthogonal_(layer.weight,math.sqrt(2));nn.init.zeros_(layer.bias)
        nn.init.orthogonal_(self.actor[-1].weight,.01);nn.init.orthogonal_(self.critic[-1].weight,1.)
    def forward(self,state,valid_actions,with_value=True):
        dist=Categorical(logits=self.actor(state).masked_fill(~valid_actions,-1e9))
        value=self.critic(state).squeeze(1) if with_value else None
        return dist,value

def policy_state(policy,g,local,bits,logits):
    mask=subset_mask(bits);count=mask.sum(1,keepdim=True)
    observed=(local*mask[:,:,None]).sum(1)/count.clamp_min(1)
    gn=((g-policy.g_mean)/policy.g_std).clamp(-10,10)
    ln=((observed-policy.l_mean)/policy.l_std).clamp(-10,10)*(count>0)
    p=logits.float().softmax(1);top=p.topk(2,dim=1).values
    entropy=-(p*p.clamp_min(1e-8).log()).sum(1,keepdim=True)/math.log(5)
    state=torch.cat([gn,ln,p,entropy,top[:,0:1]-top[:,1:2],mask.float(),count.float()/3],1)
    valid=torch.cat([torch.ones(len(g),1,dtype=torch.bool,device=g.device),~mask],1)
    return state.detach(),valid

@torch.no_grad()
def cached_inference(policy,cache,lookup,method,device):
    if method not in METHODS:raise ValueError(method)
    n=len(cache['g']);outputs=[];final_bits=[]
    for start in range(0,n,256):
        g=cache['g'][start:start+256].to(device);local=cache['local'][start:start+256].to(device)
        table=lookup[start:start+256].to(device);rows=torch.arange(len(g),device=device)
        bits=torch.zeros(len(g),dtype=torch.long,device=device)
        if method in ('DARKNESS','COLOUR','BOUNDARY','ALL_CROPS'):
            bits.fill_({'DARKNESS':1,'COLOUR':2,'BOUNDARY':4,'ALL_CROPS':7}[method])
        elif method=='CONFIDENCE':bits=(table[:,0].softmax(1).max(1).values<CFG.confidence_threshold).long()
        elif method=='PPO':
            policy.eval();alive=torch.ones(len(g),dtype=torch.bool,device=device)
            for _ in range(3):
                state,valid=policy_state(policy,g,local,bits,table[rows,bits])
                act=policy(state,valid,False)[0].probs.argmax(1)
                choose=alive & (act>0)
                bits[choose]|=1 << (act[choose]-1)
                alive &= (act!=0) & (bits!=7)
                if not alive.any():break
        outputs.append(table[rows,bits].cpu());final_bits.append(bits.cpu())
    bits=torch.cat(final_bits)
    return torch.cat(outputs),bits

@torch.no_grad()
def collect_rollout(policy,g,local,table,y):
    n=len(g);device=g.device;rows=torch.arange(n,device=device)
    bits=torch.zeros(n,dtype=torch.long,device=device);done=torch.zeros(n,dtype=torch.bool,device=device)
    records=[]
    for _ in range(3):
        before=table[rows,bits];state,mask=policy_state(policy,g,local,bits,before)
        dist,value=policy(state,mask);action=dist.sample();active=~done
        new_bits=bits.clone();choose=active & (action>0)
        new_bits[choose]|=1 << (action[choose]-1)
        after=table[rows,new_bits];terminal=(action==0)|(new_bits==7)
        delta=F.cross_entropy(before,y,reduction='none')-F.cross_entropy(after,y,reduction='none')
        reward=(delta-CFG.crop_cost*choose.float())*active
        reward+=torch.where(after.argmax(1)==y,1.,-1.)*(terminal & active)
        records.append({'state':state,'mask':mask,'action':action,'logp':dist.log_prob(action),
                        'value':value,'reward':reward,'terminal':terminal|done,'active':active})
        bits=new_bits;done|=terminal
        if done.all():break
    assert done.all(), 'Every rollout must end at STOP or the finite crop budget.'
    advantage=torch.zeros(n,device=device);next_value=torch.zeros_like(advantage)
    for r in reversed(records):
        continuation=(~r['terminal']).float()
        delta=r['reward']+CFG.gamma*next_value*continuation-r['value']
        advantage=(delta+CFG.gamma*CFG.gae_lambda*continuation*advantage)*r['active']
        r['advantage']=advantage.clone();r['return']=advantage+r['value'];next_value=r['value']
    keys=('state','mask','action','logp','advantage','return')
    data={k:torch.cat([r[k][r['active']] for r in records]).detach() for k in keys}
    data['advantage']=(data['advantage']-data['advantage'].mean())/data['advantage'].std(unbiased=False).clamp_min(1e-6)
    return data

def ppo_update(policy,opt,data):
    values=[];stop=False;n=len(data['action'])
    for _ in range(CFG.ppo_epochs):
        for ix in torch.randperm(n,device=data['state'].device).split(CFG.ppo_minibatch):
            dist,value=policy(data['state'][ix],data['mask'][ix])
            logratio=dist.log_prob(data['action'][ix])-data['logp'][ix];ratio=logratio.exp()
            with torch.no_grad():kl=((ratio-1)-logratio).mean()
            if kl>CFG.target_kl:stop=True;break
            a=data['advantage'][ix]
            actor_loss=-torch.minimum(ratio*a,ratio.clamp(1-CFG.ppo_clip,1+CFG.ppo_clip)*a).mean()
            value_loss=F.mse_loss(value,data['return'][ix])
            entropy=dist.entropy().mean()
            loss=actor_loss+CFG.value_weight*value_loss-CFG.entropy_weight*entropy
            if not torch.isfinite(loss):raise FloatingPointError('Non-finite PPO objective.')
            opt.zero_grad(set_to_none=True);loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(),CFG.grad_clip);opt.step()
            values.append([float(loss.detach()),float(kl),float(entropy.detach())])
        if stop:break
    return np.mean(values,axis=0).tolist() if values else [0.,float(kl),0.]

def policy_functions(train,val,train_lookup,val_lookup,device):
    training=cache_device(train,device);training_table=train_lookup.to(device)
    def step(model,opt,scaler,epoch):
        model.train();logs=[]
        for ix in torch.randperm(len(training['y']),device=device).split(CFG.ppo_batch_size):
            data=collect_rollout(model,training['g'][ix],training['local'][ix],training_table[ix],training['y'][ix])
            logs.append(ppo_update(model,opt,data))
        mean=np.mean(logs,axis=0)
        return {'train_loss':float(mean[0]),'approx_kl':float(mean[1]),'entropy':float(mean[2])}
    def validate(model):
        logits,bits=cached_inference(model,val,val_lookup,'PPO',device)
        return metric_values(val['y'].numpy(),logits.numpy(),subset_mask(bits).sum(1).numpy())
    return step,validate


## 10. Label-free selective inference

This is the actual deployment route: one global pass, then zero to three requested crop passes. Teacher and critic are not needed. Repeated cue actions are prohibited and STOP never performs a crop. Proposal boxes can still overlap or coincide; the diversity penalty is not a guarantee of unique regions. The same batch may contain images stopping at different steps.

A validation check compares online logits and crop choices against the cached environment for every predeclared method. A mismatch stops the study instead of silently reporting a different model.

In [ ]:
@torch.no_grad()
def online_inference(student,fusion,policy,batch,method,device):
    student.eval();fusion.eval();policy.eval()
    x,v=batch['rgb'].to(device),batch['valid'].to(device)
    g,base,maps=student(x,v);n=len(x)
    if method=='GLOBAL':return base.float(),torch.zeros(n,dtype=torch.long,device=device)
    boxes=crop_boxes(maps,x,v,batch['meta'])
    local=torch.zeros(n,3,CFG.feature_dim,device=device)
    bits=torch.zeros(n,dtype=torch.long,device=device);alive=torch.ones(n,dtype=torch.bool,device=device)
    logits=base.float().clone()
    for t in range(3):
        action=torch.zeros(n,dtype=torch.long,device=device)
        if method=='PPO':
            state,valid=policy_state(policy,g,local,bits,logits)
            action=policy(state,valid,False)[0].probs.argmax(1)
        elif method=='ALL_CROPS':action[alive]=t+1
        elif method in ('DARKNESS','COLOUR','BOUNDARY') and t==0:
            action[alive]={'DARKNESS':1,'COLOUR':2,'BOUNDARY':3}[method]
        elif method=='CONFIDENCE' and t==0:action[alive & (logits.softmax(1).max(1).values<CFG.confidence_threshold)]=1
        choose=alive & (action>0)
        if not choose.any():break
        indices=torch.where(choose)[0];acts=action[indices]
        assert not subset_mask(bits[indices])[torch.arange(len(indices),device=device),acts-1].any()
        cx,cv=crops_from_batch(batch,boxes,indices.cpu().tolist(),acts.cpu().tolist(),device)
        z,_,_=student(cx,cv);local[indices,acts-1]=z.float();bits[indices]|=1 << (acts-1)
        logits[indices]=fusion(g[indices],base[indices].float(),local[indices],subset_mask(bits[indices]))
        alive &= (action!=0) & (bits!=7)
    return logits,bits

@torch.no_grad()
def check_online_cache(ctx,student,fusion,policy,val,val_lookup):
    b=next(iter(loader(ctx['val'],batch_size=min(4,CFG.batch_size))));n=len(b['ids'])
    small={k:v[:n] for k,v in val.items()};results={}
    for method in METHODS:
        actual,bits=online_inference(student,fusion,policy,b,method,ctx['device'])
        expected,eb=cached_inference(policy,small,val_lookup[:n],method,ctx['device'])
        diff=float((actual.cpu()-expected).abs().max())
        if diff>2e-3 or not torch.equal(bits.cpu(),eb):raise AssertionError(f'Online/cache mismatch for {method}: {diff}')
        results[method]={'max_logit_difference':diff,'route_equal':True}
    return results


## 11. Metrics, confusion matrices and diagnostic headroom

Every method gets accuracy, macro-F1, balanced accuracy, CE, per-class metrics, prediction CSV, confusion CSV/PNG and mean crop count.

The label-aware oracle checks whether **any** observed subset could correctly classify an image. It is only an upper-bound diagnostic using true labels; it is never a policy result and never selects per-image deployment actions.

The reports ZIP contains JSON/CSV/figures only. It **does not** contain weights or caches. The full output folder is required to resume.

In [ ]:
def report_predictions(frame,logits,bits,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    y=frame.label.to_numpy();z=logits.detach().cpu().numpy() if isinstance(logits,torch.Tensor) else np.asarray(logits)
    p=torch.tensor(z).softmax(1).numpy();pred=p.argmax(1)
    bit=np.asarray(bits,dtype=int);counts=np.array([int(b).bit_count() for b in bit])
    m=metric_values(y,z,counts);save_json(m,out/'metrics.json')
    table=frame[['sample_id','relative_path','group_id','class_name','label']].copy()
    table['prediction']=pred;table['predicted_class']=[CLASSES[i] for i in pred]
    table['confidence']=p.max(1);table['crop_subset_bits']=bit;table['extra_crops']=counts
    for i,c in enumerate(CLASSES):table['p_'+c]=p[:,i]
    table.to_csv(out/'predictions.csv',index=False)
    cr=classification_report(y,pred,labels=list(range(5)),target_names=CLASSES,output_dict=True,zero_division=0)
    pd.DataFrame(cr).T.to_csv(out/'classification_report.csv')
    cm=confusion_matrix(y,pred,labels=list(range(5)))
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(out/'confusion_matrix.csv')
    fig,ax=plt.subplots(figsize=(7,6));ax.imshow(cm,cmap='Blues')
    ax.set(xticks=range(5),yticks=range(5),xticklabels=CLASSES,yticklabels=CLASSES,xlabel='Predicted',ylabel='True')
    plt.setp(ax.get_xticklabels(),rotation=40,ha='right')
    for i in range(5):
        for j in range(5):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
    ax.set_title(f'Accuracy {m["accuracy"]:.2%} | macro-F1 {m["macro_f1"]:.2%}')
    fig.tight_layout();fig.savefig(out/'confusion_matrix.png',dpi=140);plt.close(fig)
    return m

def oracle_headroom(cache,lookup):
    y=cache['y'];correct=lookup.argmax(2)==y[:,None];base=correct[:,0]
    rows=[]
    for bit in range(8):
        rows.append({'subset_bits':bit,'crops':bit.bit_count(),'accuracy':float(correct[:,bit].float().mean()),
                     'rescued':int((~base & correct[:,bit]).sum()),'broken':int((base & ~correct[:,bit]).sum())})
    return rows,{'global_accuracy':float(base.float().mean()),'label_aware_any_subset_accuracy':float(correct.any(1).float().mean()),
                'scope':'Diagnostic upper bound using true labels, NOT an achievable policy accuracy or deployable router.'}

@torch.no_grad()
def attention_examples(ctx,student,seed):
    student.eval();b=next(iter(loader(ctx['val'],batch_size=3)))
    x,v=b['rgb'].to(ctx['device']),b['valid'].to(ctx['device'])
    _,_,maps=student(x,v);heat=F.interpolate(maps.float(),(CFG.image_size,CFG.image_size),mode='bilinear',align_corners=False).cpu()
    fig,axes=plt.subplots(len(x),4,figsize=(12,3*len(x)),squeeze=False)
    for i in range(len(x)):
        im=b['rgb'][i].permute(1,2,0).numpy()
        for j in range(4):
            axes[i,j].imshow(im);axes[i,j].axis('off')
            if j:axes[i,j].imshow(heat[i,j-1].numpy(),cmap='magma',alpha=.5)
            axes[i,j].set_title(('Input','Darkness query','Colour query','Boundary query')[j])
    fig.suptitle('Validation attention examples: image-derived cues, not anatomical segmentation')
    fig.tight_layout();fig.savefig(ctx['root']/f'seed_{seed}'/'attention_examples.png',dpi=130);plt.close(fig)

def summarize(rows,path):
    table=pd.DataFrame(rows);table.to_csv(path,index=False)
    summary=table.groupby('method').agg(seeds=('seed','nunique'),accuracy_mean=('accuracy','mean'),
          accuracy_seed_sd=('accuracy',lambda x:float(x.std(ddof=1)) if len(x)>1 else 0.),
          macro_f1_mean=('macro_f1','mean'),macro_f1_seed_sd=('macro_f1',lambda x:float(x.std(ddof=1)) if len(x)>1 else 0.),
          mean_crops=('mean_crops','mean')).reset_index()
    summary.to_csv(Path(path).with_name(Path(path).stem+'_summary.csv'),index=False)
    return summary

def export_reports(ctx):
    root=ctx['root'];path=root/'lightcyto_fresh_reports.zip'
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(root.rglob('*')):
            if p.is_file() and p.suffix.lower() in {'.json','.csv','.png','.svg','.md'}:
                z.write(p,p.relative_to(root))
    return path


## 12. Deployment and latency

Each seed exports a `deployment/model.pt` for the **validation-selected** method. The package omits teacher weights and critic weights; global-only deployment also omits fusion and actor weights. `predict_image()` is a label-free entry point.

Latency is measured with actual online crop selection, GPU synchronization, warm-up and batch size one, on decoded validation images. Disk decoding is excluded; native crop creation and encoder calls are included. Global, PPO, all-crop and selected-method timings are recorded. TinyViT relative-position bias caches are cleared after checkpoint reloads.

Parameter count and file size are measured, not inferred from the backbone model-card number. No fake claim of 4× speed-up is made from cached features.

In [ ]:
class ActorOnly(Policy):
    def __init__(self):
        super().__init__();del self.critic
    def forward(self,state,valid_actions,with_value=False):
        if with_value:raise ValueError('The deployed actor has no critic.')
        return Categorical(logits=self.actor(state).masked_fill(~valid_actions,-1e9)),None

def export_deployment(ctx,seed,student,fusion,policy,method):
    out=ctx['root']/f'seed_{seed}'/'deployment';out.mkdir(parents=True,exist_ok=True)
    actor_state={k:v for k,v in cpu_state(policy).items() if not k.startswith('critic.')}
    weights={'student':cpu_state(student)}
    if method!='GLOBAL':weights['fusion']=cpu_state(fusion)
    if method=='PPO':weights['actor']=actor_state
    path=out/'model.pt'
    save_pt({'weights':weights,'method':method,'classes':list(CLASSES),'config':asdict(CFG),
             'contract':ctx['contract_hash'],'version':VERSION},path)
    params=sum(p.numel() for p in student.parameters())
    if method!='GLOBAL':params+=sum(p.numel() for p in fusion.parameters())
    if method=='PPO':params+=sum(p.numel() for p in policy.actor.parameters())
    save_json({'method':method,'deploy_parameters':params,'checkpoint_megabytes':path.stat().st_size/1e6,
               'teacher_included':False,'critic_included':False,'student_pretraining':'ImageNet22K distillation then ImageNet1K',
               'ensemble':False,'tta':False},out/'deployment.json')

@torch.no_grad()
def benchmark(ctx,student,fusion,policy,methods):
    if CFG.benchmark_images<=0:return []
    frame=ctx['val'].head(CFG.benchmark_images);batches=list(loader(frame,batch_size=1));device=ctx['device']
    def sync():
        if device.type=='cuda':torch.cuda.synchronize()
    rows=[]
    # Batch size one; decoded raw RGB images are already resident. Crop extraction is timed.
    for method in methods:
        for _ in range(2):online_inference(student,fusion,policy,batches[0],method,device)
        times=[];counts=[]
        for b in batches:
            sync();start=time.perf_counter();_,bits=online_inference(student,fusion,policy,b,method,device);sync()
            times.append((time.perf_counter()-start)*1000);counts.append(int(bits[0]).bit_count())
        rows.append({'method':method,'images':len(times),'batch_size':1,'mean_ms':float(np.mean(times)),
                     'median_ms':float(np.median(times)),'p95_ms':float(np.percentile(times,95)),
                     'mean_crops':float(np.mean(counts)),'mean_encoder_passes':1+float(np.mean(counts)),
                     'scope':'Online selective inference on decoded validation images; disk decoding excluded; FP32.'})
    return rows

def load_deployment(path,device=None):
    bundle=load_pt(path)
    if bundle['version']!=VERSION:raise ValueError('Deployment architecture version mismatch.')
    if bundle['classes']!=list(CLASSES):raise ValueError('Class order mismatch.')
    # Architecture and preprocessing choices must match the saved recipe.
    for k in ('image_size','feature_dim','attention_heads','crop_fraction','overlap_penalty','confidence_threshold','max_crops'):
        if bundle['config'][k]!=getattr(CFG,k):raise ValueError('Restore saved CFG.'+k+' before inference.')
    device=torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    student=Student(False).to(device);install_state(student,bundle['weights']['student']);freeze(student)
    fusion=Fusion().to(device);actor=ActorOnly().to(device)
    if 'fusion' in bundle['weights']:install_state(fusion,bundle['weights']['fusion'])
    if 'actor' in bundle['weights']:install_state(actor,bundle['weights']['actor'])
    return {'student':student,'fusion':freeze(fusion),'policy':freeze(actor),'method':bundle['method'],'device':device}

@torch.no_grad()
def predict_image(image_path,deployment):
    with Image.open(image_path) as im:im=im.convert('RGB')
    rgb,valid,meta=letterbox(im,CFG.image_size)
    batch={'rgb':rgb[None],'valid':valid[None],'raw':[np.asarray(im).copy()],'meta':[meta]}
    logits,bits=online_inference(deployment['student'],deployment['fusion'],deployment['policy'],batch,
                                 deployment['method'],deployment['device'])
    p=logits.softmax(1)[0].cpu().tolist()
    return {'class':CLASSES[int(np.argmax(p))],'probabilities':dict(zip(CLASSES,p)),
            'extra_crops':int(bits[0]).bit_count(),'method':deployment['method']}


## 13. Full study and automatic final testing

Order: train shared teacher → for each declared seed train student → cache train/validation features → train fusion → train PPO → compare validation methods → lock one common method across seeds → evaluate test → export deployment.

Predeclared comparisons: global-only, each single fixed crop, all three crops, fixed confidence routing (threshold 0.80, one darkness crop), and PPO. The primary deployment method is selected by **mean validation macro-F1**, then accuracy, then fewer crops. All seven test comparisons are reported; no method or seed is chosen by its test result.

Test predictions use an all-crop feature cache for efficient counterfactual comparison; policy observation masks remain enforced. **That cache's extraction time is not the selective deployment runtime.** Online/cache parity is checked separately and actual selective latency is benchmarked above.

One seed gives one model's test metrics. Multiple seeds give per-seed metrics and mean/SD on the same unique test images—not an ensemble and not extra independent test samples. Final-test completion is cached and tied to locked checkpoint hashes.

In [ ]:
def train_one_seed(ctx,seed,teacher):
    device=ctx['device'];out=ctx['root']/f'seed_{seed}';out.mkdir(exist_ok=True)
    seed_all(seed)
    student=Student(CFG.pretrained and not (out/'student'/'last.pt').exists()).to(device)
    student=fit_stage(ctx,f'seed_{seed}/student',student,CFG.student_epochs,CFG.student_lr,seed,
        image_train_epoch(ctx,False,teacher),lambda m:image_validation(m,ctx['val'],device),CFG.amp and device.type=='cuda')
    freeze(student)
    train=build_cache(ctx,student,seed,'train');val=build_cache(ctx,student,seed,'val')
    seed_all(seed+100);fusion=Fusion().to(device);step,validate=fusion_functions(train,val,device)
    fit_stage(ctx,f'seed_{seed}/fusion',fusion,CFG.fusion_epochs,CFG.fusion_lr,seed+100,step,validate)
    freeze(fusion);train_lookup=subset_logits(fusion,train,device);val_lookup=subset_logits(fusion,val,device)
    rows,headroom=oracle_headroom(val,val_lookup)
    pd.DataFrame(rows).to_csv(out/'validation_subset_headroom.csv',index=False);save_json(headroom,out/'validation_oracle.json')
    seed_all(seed+200);policy=Policy(train).to(device)
    step,validate=policy_functions(train,val,train_lookup,val_lookup,device)
    fit_stage(ctx,f'seed_{seed}/ppo',policy,CFG.ppo_rounds,CFG.ppo_lr,seed+200,step,validate)
    freeze(policy)
    save_json(check_online_cache(ctx,student,fusion,policy,val,val_lookup),out/'online_cache_check.json')
    results=[]
    for method in METHODS:
        logits,bits=cached_inference(policy,val,val_lookup,method,device)
        m=report_predictions(ctx['val'],logits,bits,out/'validation'/method)
        results.append({'seed':seed,'method':method,**m})
    attention_examples(ctx,student,seed)
    del student,fusion,policy,train,val,train_lookup,val_lookup;gc.collect()
    if device.type=='cuda':torch.cuda.empty_cache()
    return results

def design_lock(ctx,rows):
    summary=summarize(rows,ctx['root']/'validation_results.csv')
    # One common method across all declared seeds, chosen before test access.
    ranking=summary.sort_values(['macro_f1_mean','accuracy_mean','mean_crops','method'],ascending=[False,False,True,True])
    selected=str(ranking.iloc[0]['method'])
    hashes={}
    for seed in CFG.seeds:
        for stage in ('student','fusion','ppo'):
            p=ctx['root']/f'seed_{seed}'/stage/'best.pt';hashes[str(p.relative_to(ctx['root']))]=file_hash(p)
    lock={'contract':ctx['contract_hash'],'selected_method':selected,'selection':'Mean validation macro-F1, then accuracy, then fewer crops',
          'seeds':list(CFG.seeds),'predeclared_test_methods':list(METHODS),'checkpoint_sha256':hashes,
          'scope':'Post-selection follow-up; earlier results on SIPaKMeD informed this recipe.'}
    p=ctx['root']/'design_lock.json'
    if p.exists() and read_json(p)!=lock:raise ValueError('An existing design lock differs; do not overwrite test-selected decisions.')
    save_json(lock,p);print('Validation-selected deployment method:',selected,flush=True)
    return lock

def load_seed(ctx,seed):
    device=ctx['device'];out=ctx['root']/f'seed_{seed}'
    student=Student(False).to(device);fusion=Fusion().to(device);policy=Policy().to(device)
    for model,stage in ((student,'student'),(fusion,'fusion'),(policy,'ppo')):
        ck=load_pt(out/stage/'best.pt')
        if ck['contract']!=ctx['contract_hash']:raise ValueError('Checkpoint identity mismatch.')
        install_state(model,ck['model']);freeze(model)
    return student,fusion,policy

def final_test(ctx,lock):
    if not CFG.run_final_test:return {'status':'disabled_by_config'}
    for rel,h in lock['checkpoint_sha256'].items():
        if file_hash(ctx['root']/rel)!=h:raise ValueError('A locked checkpoint changed: '+rel)
    out=ctx['root']/'final_test';out.mkdir(exist_ok=True)
    done=out/'complete.json'
    if done.exists():
        result=read_json(done)
        if result['design_lock_hash']!=object_hash(lock):raise ValueError('Test lock mismatch.')
        print('Final test already complete; keeping the saved results.',flush=True);return result
    all_rows=[]
    for seed in CFG.seeds:
        budget(ctx);student,fusion,policy=load_seed(ctx,seed)
        test=build_cache(ctx,student,seed,'test');lookup=subset_logits(fusion,test,ctx['device'])
        for method in METHODS:
            logits,bits=cached_inference(policy,test,lookup,method,ctx['device'])
            m=report_predictions(ctx['test'],logits,bits,out/f'seed_{seed}'/method)
            all_rows.append({'seed':seed,'method':method,'primary':method==lock['selected_method'],**m})
        export_deployment(ctx,seed,student,fusion,policy,lock['selected_method'])
        timing=benchmark(ctx,student,fusion,policy,list(dict.fromkeys(['GLOBAL','PPO','ALL_CROPS',lock['selected_method']])))
        pd.DataFrame(timing).to_csv(ctx['root']/f'seed_{seed}'/'deployment'/'online_latency.csv',index=False)
        del student,fusion,policy,test,lookup;gc.collect()
        if ctx['device'].type=='cuda':torch.cuda.empty_cache()
    summary=summarize(all_rows,out/'test_results.csv')
    selected=summary[summary.method==lock['selected_method']].iloc[0].to_dict()
    result={'status':'complete','design_lock_hash':object_hash(lock),'primary':selected,'unique_test_images':len(ctx['test']),
            'aggregation':'Single-model metrics per seed; mean over seeds if multiple; NOT an ensemble.',
            'target_met':bool(selected['accuracy_mean']>=.97 and selected['macro_f1_mean']>=.97)}
    save_json(result,done);return result

def run_pipeline(ctx):
    teacher=None
    try:
        seed_all(CFG.teacher_seed);device=ctx['device']
        teacher=new_teacher(CFG.pretrained and not (ctx['root']/'teacher'/'last.pt').exists()).to(device)
        fit_stage(ctx,'teacher',teacher,CFG.teacher_epochs,CFG.teacher_lr,CFG.teacher_seed,image_train_epoch(ctx,True),
                  lambda m:image_validation(m,ctx['val'],device,True),CFG.amp and device.type=='cuda')
        freeze(teacher);results=[]
        for seed in CFG.seeds:results.extend(train_one_seed(ctx,seed,teacher))
        teacher.to('cpu');del teacher;teacher=None;gc.collect()
        if device.type=='cuda':torch.cuda.empty_cache()
        lock=design_lock(ctx,results);test=final_test(ctx,lock)
        status={'status':'complete' if test.get('status')=='complete' else 'training_complete_test_disabled','final_test':test}
        save_json(status,ctx['root']/'session_status.json');return status
    except SessionPause as e:
        status={'status':'paused','reason':str(e),'next_step':'Save FULL Kaggle output; attach it next session and set CFG.resume_source.'}
        save_json(status,ctx['root']/'session_status.json');print(status,flush=True);return status
    except Exception as e:
        save_json({'status':'error','type':type(e).__name__,'reason':str(e)},ctx['root']/'session_status.json');raise
    finally:
        report=export_reports(ctx);print('Reports:',report,'\nFULL resumable output:',ctx['root'],flush=True)


## 14. Figures for the team
The notebook saves a pipeline diagram, validation attention examples and learning curves. Attention plots are explanatory visualizations, not evidence of anatomically correct segmentation.

In [ ]:
def architecture_figure(root):
    from matplotlib.patches import FancyBboxPatch
    fig,ax=plt.subplots(figsize=(12,8));ax.set(xlim=(0,12),ylim=(0,8));ax.axis('off')
    def box(x,y,w,h,text,color='#E5F2FA'):
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.08',facecolor=color,edgecolor='#235778'))
        ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=10)
    def arrow(a,b,dashed=False):ax.annotate('',xy=b,xytext=a,arrowprops={'arrowstyle':'->','color':'#235778','linestyle':'--' if dashed else '-'})
    box(.3,6.7,2.3,.65,'RGB cell image');box(3.1,6.5,3.3,1.,'TinyViT 5M\n192-channel projection\n3-query / 4-head attention')
    box(7,6.6,2.6,.8,'Global features +\n5-class prediction')
    arrow((2.7,7.0),(3,7));arrow((6.5,7),(6.9,7))
    box(.3,4.7,2.3,1.,'Image-derived cues\nDarkness / colour / boundary');arrow((1.45,6.6),(1.45,5.8));arrow((2.7,5.3),(3.5,6.4))
    box(7,4.8,2.6,1.,'PPO actor\nObserved evidence +\nremaining budget');arrow((8.3,6.5),(8.3,5.9))
    box(10,4.8,1.6,1.,'STOP\nPrediction');arrow((9.7,5.3),(9.9,5.3))
    box(7,2.7,2.6,1.,'Selected native crop\nShared student encoder');arrow((8.3,4.7),(8.3,3.8))
    box(3.1,2.7,3.3,1.,'Residual global-local fusion\nUpdate prediction and state');arrow((6.9,3.2),(6.5,3.2));arrow((6,3.8),(7.2,4.7))
    arrow((7,6.5),(5,3.8))
    box(.3,.6,3.0,1.1,'Training only\nConvNeXt-Tiny teacher\nCE + KD supervision','#FFF0D9')
    arrow((2,1.8),(3.6,6.4),True)
    ax.text(4,.7,'Teacher and critic are omitted at deployment.\nPPO chooses at most 3 distinct cue actions.\nFinal deployment method is selected on validation.',fontsize=11,color='#235778')
    ax.set_title('LightCyto: fresh TinyViT + custom attention + KD + PPO',fontsize=16,pad=12)
    fig.tight_layout();root=Path(root);fig.savefig(root/'architecture.png',dpi=150);fig.savefig(root/'architecture.svg');return fig

def plot_histories(ctx):
    files=list(ctx['root'].rglob('history.csv'))
    for p in files:
        df=pd.read_csv(p)
        if not len(df):continue
        fig,axes=plt.subplots(1,2,figsize=(10,3))
        axes[0].plot(df.epoch,df.train_loss);axes[0].set(title='Training objective',xlabel='Epoch / PPO round')
        axes[1].plot(df.epoch,df.accuracy,label='Validation accuracy');axes[1].plot(df.epoch,df.macro_f1,label='Validation macro-F1')
        axes[1].legend();axes[1].set(xlabel='Epoch / PPO round',ylim=(0,1))
        fig.suptitle(str(p.parent.relative_to(ctx['root'])));fig.tight_layout();fig.savefig(p.parent/'learning_curves.png',dpi=130);plt.close(fig)


## 15. Run from the beginning / resume pending work

This is the execution cell. With the default settings, **Run All includes final testing** once all planned stages finish. If paused, continue with the same code, runtime and settings; completed checkpoints and feature batches are reused.

**Nothing is trained on the historical v4/v5 reports.** No pretrained task checkpoint is needed for a fresh run.

In [ ]:
from IPython.display import display, FileLink
CONTEXT = prepare()
display(pd.read_csv(CONTEXT['root']/'split_summary.csv'))
fig = architecture_figure(CONTEXT['root'])
display(fig)
plt.close(fig)
RESULT = run_pipeline(CONTEXT)
plot_histories(CONTEXT)
REPORT_ZIP = export_reports(CONTEXT)
print(json.dumps(RESULT, indent=2))
display(FileLink(str(REPORT_ZIP)))


## 16. Read the result

Look at `session_status.json` first. A paused study has not finished testing. `final_test/complete.json` identifies the preselected primary method and whether **both** test accuracy and macro-F1 meet 0.97. Report the measured outcome even if the target is missed.

Default main outputs are under `/kaggle/working/lightcyto_tinyvit_end_to_end/`:

| File/folder | What it contains |
|---|---|
| `manifest.csv`, `data_audit.json`, `split_summary.csv` | Exact partition and grouping audit |
| `teacher/` | Teacher best/last checkpoints and training history |
| `seed_42/student/`, `fusion/`, `ppo/` | Best/last checkpoints for each stage |
| `seed_42/cache/` | Newly extracted frozen features |
| `validation_results_summary.csv` | Validation comparison |
| `design_lock.json` | Method chosen before test prediction |
| `final_test/test_results_summary.csv` | Test accuracy/F1 by method, mean across declared seeds |
| `final_test/seed_42/<METHOD>/` | Prediction CSV, per-class metrics and confusion matrix |
| `seed_42/deployment/` | Selected model, parameter count, storage and actual online latency |
| `lightcyto_fresh_reports.zip` | Small archive to upload for analysis |

The seed folder changes with the configured seed(s). No 97.61% or other historical score is hardcoded as a new result.

In [ ]:
for relative in ('validation_results_summary.csv','final_test/test_results_summary.csv'):
    path = CONTEXT['root']/relative
    if path.exists():
        print(relative)
        display(pd.read_csv(path))
    else:
        print('Pending:', relative)
print('Status:', read_json(CONTEXT['root']/'session_status.json')['status'])
print('Save FULL output for resume:', CONTEXT['root'])
display(FileLink(str(CONTEXT['root']/'lightcyto_fresh_reports.zip')))


## 17. Resume in another Kaggle session

1. Save the **full** output as a Kaggle dataset or notebook output, including teacher/student/fusion/PPO checkpoints and `cache/`.
2. Attach that saved output and the original SIPaKMeD images to the next session.
3. Set `CFG.resume_source` to the exact saved `lightcyto_tinyvit_end_to_end` directory. Keep `output_root` writable under `/kaggle/working`.
4. Keep scientific settings and definitions unchanged. Run All again. It restores to the writable folder and runs pending stages.

For example, in the configuration cell:

```python
CFG = Config(
    dataset_root='',
    resume_source='/kaggle/input/YOUR-SAVED-DATASET/lightcyto_tinyvit_end_to_end',
    output_root='/kaggle/working/lightcyto_tinyvit_end_to_end',
)
```

Use your real path, not the example placeholder. The reports ZIP alone cannot resume training. New seeds or a new resolution/budget are a new experiment, so use a new output directory. Changing timm/torch or the implementation is detected before reusing the run.

## 18. Predict one image after completion

This optional cell is disabled until you supply an image path. It uses the exported validation-selected method, including PPO if PPO was selected. It requires the definition cells above and the same architecture/preprocessing configuration, but no teacher or training dataset.

In [ ]:
IMAGE_TO_PREDICT = ''
if IMAGE_TO_PREDICT:
    model_path = Path(CFG.output_root)/f'seed_{CFG.seeds[0]}'/'deployment'/'model.pt'
    deployed = load_deployment(model_path)
    print(json.dumps(predict_image(IMAGE_TO_PREDICT, deployed), indent=2))


## 19. Validation and reporting limits

Software checks for this handoff used PyTorch 2.14.1+cpu, torchvision 0.29.1+cpu and timm 1.0.22 with **random initialization and synthetic images**. Checks covered the actual ConvNeXt-Tiny and TinyViT architectures through teacher training, KD student training, fusion, PPO, final test and export; online/cached prediction agreement; unseen-crop masking; STOP behavior; terminal GAE; source-field duplicate grouping; and exact CPU epoch-boundary AdamW recovery.

These checks establish software behavior, **not SIPaKMeD accuracy**. The full dataset, official pretrained downloads, CUDA mixed precision, Kaggle runtime and GPU latency must be exercised in your run. Runtime versions are recorded and must match when resuming that run. No expected training duration is invented; stage estimates use measured epoch time.

Use accuracy, macro-F1, per-class recall, number of unique test images, source-group protocol, seeds, pretraining sources, active inference method and actual compute in your report. PPO was trained even if validation ultimately selects global-only. A successful run does not by itself establish attention novelty, a clinical claim or a journal acceptance level.

Sources:
- [TinyViT paper](https://arxiv.org/abs/2207.10666) and [official timm model card](https://huggingface.co/timm/tiny_vit_5m_224.dist_in22k_ft_in1k)
- [ConvNeXt-Tiny implementation and weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.convnext_tiny.html)
- [Knowledge distillation](https://arxiv.org/abs/1503.02531)
- [PPO](https://arxiv.org/abs/1707.06347) and [GAE](https://arxiv.org/abs/1506.02438)
- [Implementation Matters](https://arxiv.org/abs/2005.12729) and [What Matters in On-Policy RL](https://arxiv.org/abs/2006.05990)
- [SIPaKMeD dataset](https://www.cs.uoi.gr/~marina/sipakmed.html)